In [4]:
import gdown
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from scipy.sparse import issparse
from torch.utils.data import Dataset
import random
import time
import os
import csv
import sys

def load_splits_from_drive(folder_url, output_dir):
    """
    Download and load train/val/test splits from a Google Drive folder.
    Skips download if all split files already exist locally.
    The folder must be shared with 'Anyone with the link can view'.
    """
    os.makedirs(output_dir, exist_ok=True)

    expected_files = ["train.pkl", "val.pkl", "test.pkl"]
    existing = [f for f in expected_files 
                if os.path.exists(os.path.join(output_dir, f))]

    if len(existing) == len(expected_files):
        print("All split files already exist locally, skipping download.")
    else:
        if existing:
            missing = set(expected_files) - set(existing)
            print(f"Missing files: {missing}, re-downloading...")
        else:
            print("No split files found, downloading from Google Drive...")
        gdown.download_folder(folder_url, output=output_dir, quiet=False)

    print("\nLoading splits...")
    train_df = pd.read_pickle(os.path.join(output_dir, "train.pkl"))
    val_df   = pd.read_pickle(os.path.join(output_dir, "val.pkl"))
    test_df  = pd.read_pickle(os.path.join(output_dir, "test.pkl"))

    print(f"  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
    return train_df, val_df, test_df

In [2]:
FOLDER_URL = "https://drive.google.com/drive/folders/1kSe6d4Kh3aUUP0x3l_O2L9AvperSZocA?usp=sharing"
OUTPUT_DIR = r"C:\Users\Kian\Documents\GitHub\GDSProject\splits"

train_df, val_df, test_df = load_splits_from_drive(FOLDER_URL, OUTPUT_DIR)

All split files already exist locally, skipping download.

Loading splits...
  Train: 796,000 | Val: 99,500 | Test: 99,500


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Neural Network Fake News Classifier
# ═══════════════════════════════════════════════════════════════════════════════
#
# Approach: TF-IDF (top 50k features) → 2-layer feedforward neural network
#
# Why this over alternatives?
#   - TF-IDF over raw bag-of-words: downweights ubiquitous words (e.g. "said",
#     "one") that dominate BoW but carry little discriminative signal. This gives
#     the model a meaningfully better starting representation.
#   - Neural network over Logistic Regression: can learn non-linear combinations
#     of TF-IDF features — e.g. co-occurrence patterns across topics — that a
#     linear boundary cannot capture.
#   - Neural network over SVM: scales better to large vocabularies and large
#     datasets (mini-batch SGD vs. full-kernel matrix), and dropout regularisation
#     is simpler to tune than SVM's C/gamma.
#   - Neural network over Naive Bayes: NB assumes feature independence, which is
#     a poor fit for text where word combinations carry meaning.
#
# Architecture:
#   Input (50,000)  →  Dense(512, ReLU) + BN + Dropout(0.4)
#                   →  Dense(128, ReLU) + BN + Dropout(0.3)
#                   →  Dense(1, Sigmoid)
#
# Reproducibility:
#   All random seeds are fixed (Python, NumPy, PyTorch).
#   All hyperparameters are defined as named constants at the top.
#   The TF-IDF vocabulary is fit ONLY on the training set.
#   Early stopping on validation loss prevents overfitting.
# ═══════════════════════════════════════════════════════════════════════════════
# ── Hyperparameters ───────────────────────────────────────────────────────────
VOCAB_SIZE    = 30_000   # TF-IDF max features
HIDDEN_1      = 512      # neurons in first hidden layer
HIDDEN_2      = 128      # neurons in second hidden layer
DROPOUT_1     = 0.4      # dropout rate after hidden layer 1
DROPOUT_2     = 0.3      # dropout rate after hidden layer 2
BATCH_SIZE    = 512      # mini-batch size for SGD
LR            = 1e-3     # Adam learning rate
WEIGHT_DECAY  = 1e-5     # L2 regularisation
MAX_EPOCHS    = 20       # maximum training epochs
PATIENCE      = 3        # early stopping patience (epochs without val improvement)
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
print(f"Using device: {DEVICE}")

if __name__ == "__main__":
    
    
    # ── Reproducibility seeds ─────────────────────────────────────────────────────
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    
    
    
    
    # ── Label mapping ─────────────────────────────────────────────────────────────
    # The dataset uses a 'type' column with string labels. We collapse to binary:
    #   real  →  1    (reliable, mostly true)
    #   fake  →  0    (fake, satire, bias, conspiracy, hate, junksci, unreliable)
    REAL_LABELS = {"reliable", "mostly true"}
    
    def binarize_labels(series):
        return (series.str.strip().str.lower().isin(REAL_LABELS)).astype(int)
    
    
    # ── Feature extraction ────────────────────────────────────────────────────────
    # Join stemmed tokens back into a string for scikit-learn's TfidfVectorizer.
    # Fitting is done ONLY on the training set to prevent data leakage.
    
    def tokens_to_string(token_list):
        """Convert a list of tokens to a single space-joined string."""
        if isinstance(token_list, list):
            return " ".join(token_list)
        return ""
    
    print("Building TF-IDF features...")
    t0 = time.time()
    
    train_texts = train_df["tokens_stemmed"].apply(tokens_to_string)
    val_texts   = val_df["tokens_stemmed"].apply(tokens_to_string)
    test_texts  = test_df["tokens_stemmed"].apply(tokens_to_string)
    
    # sublinear_tf=True applies log(1+tf) instead of raw tf, compressing very
    # common terms further. min_df=5 removes extremely rare noise tokens.
    tfidf = TfidfVectorizer(
        max_features = VOCAB_SIZE,
        sublinear_tf = True,
        min_df       = 10,
        ngram_range  = (1, 1),   # unigrams + bigrams for richer context
    )
    
    X_train = tfidf.fit_transform(train_texts)   # fit only on train
    X_val   = tfidf.transform(val_texts)
    X_test  = tfidf.transform(test_texts)
    
    y_train = binarize_labels(train_df["type"])
    y_val   = binarize_labels(val_df["type"])
    y_test  = binarize_labels(test_df["type"])
    
    print(f"  TF-IDF done in {time.time()-t0:.1f}s")
    print(f"  Feature matrix: {X_train.shape[0]:,} train | "
          f"{X_val.shape[0]:,} val | {X_test.shape[0]:,} test | "
          f"{X_train.shape[1]:,} features")
    print(f"  Label balance — train: "
          f"{y_train.mean()*100:.1f}% real | "
          f"val: {y_val.mean()*100:.1f}% real")
    
    
    # ── Dataset helpers ───────────────────────────────────────────────────────────
    
    # def sparse_to_tensor(X, y):
    #     """Convert a scipy sparse matrix + label Series to a TensorDataset."""
    #     if issparse(X):
    #         X = X.toarray()
    #     X_t = torch.tensor(X, dtype=torch.float32)
    #     y_t = torch.tensor(y.values, dtype=torch.float32)
    #     return TensorDataset(X_t, y_t)
    
    # print("\nConverting to tensors (this may take a moment for 50k features)...")
    # t0 = time.time()
    # train_ds = sparse_to_tensor(X_train, y_train)
    # val_ds   = sparse_to_tensor(X_val,   y_val)
    # test_ds  = sparse_to_tensor(X_test,  y_test)
    # print(f"  Done in {time.time()-t0:.1f}s")
    
    # train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
    #                           num_workers=0, pin_memory=DEVICE.type == "cuda")
    # val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    # test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
    class SparseDataset(Dataset):
        """Wraps a scipy sparse matrix — slices rows on demand, never densifies."""
        def __init__(self, X, y):
            self.X = X
            self.y = torch.tensor(y.values, dtype=torch.float32)
    
        def __len__(self):
            return self.X.shape[0]
    
        def __getitem__(self, idx):
            # .toarray() on a single row is cheap — just 30,000 floats
            x = torch.tensor(self.X[idx].toarray().squeeze(0), dtype=torch.float32)
            return x, self.y[idx]
    
            train_ds = SparseDataset(X_train, y_train)
            val_ds   = SparseDataset(X_val,   y_val)
            test_ds  = SparseDataset(X_test,  y_test)
            
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
            val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

            print("Datasets ready (sparse — no densification).")
    
    
    # ── Model definition ──────────────────────────────────────────────────────────
    
    class FakeNewsNet(nn.Module):
        """
        Two-layer feedforward network for binary fake-news classification.
    
        BatchNorm1d is placed before the activation to stabilise training on
        high-dimensional sparse TF-IDF input. Dropout after each activation
        prevents co-adaptation of neurons.
        """
    
        def __init__(self, input_dim, hidden1, hidden2, drop1, drop2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden1),
                nn.BatchNorm1d(hidden1),
                nn.ReLU(),
                nn.Dropout(drop1),
    
                nn.Linear(hidden1, hidden2),
                nn.BatchNorm1d(hidden2),
                nn.ReLU(),
                nn.Dropout(drop2),
    
                nn.Linear(hidden2, 1),
                nn.Sigmoid(),
            )
    
        def forward(self, x):
            return self.net(x).squeeze(1)
    
    
    model = FakeNewsNet(
        input_dim = VOCAB_SIZE,
        hidden1   = HIDDEN_1,
        hidden2   = HIDDEN_2,
        drop1     = DROPOUT_1,
        drop2     = DROPOUT_2,
    ).to(DEVICE)
    
    print(f"\nModel architecture:\n{model}")
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {total_params:,}")
    
    
    # ── Training ──────────────────────────────────────────────────────────────────
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    
    # Reduce LR by 0.5 if val loss plateaus for 2 consecutive epochs
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2,
    )
    
    def run_epoch(loader, train=True):
        """Run one epoch; return average loss and accuracy."""
        model.train(train)
        total_loss, correct, total = 0.0, 0, 0
    
        with torch.set_grad_enabled(train):
            for X_batch, y_batch in loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                preds = model(X_batch)
                loss  = criterion(preds, y_batch)
    
                if train:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
    
                total_loss += loss.item() * len(y_batch)
                correct    += ((preds >= 0.5).float() == y_batch).sum().item()
                total      += len(y_batch)
    
        return total_loss / total, correct / total
    
    
    print("\nTraining...")
    best_val_loss   = float("inf")
    epochs_no_improv = 0
    history          = []
    
    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = run_epoch(train_loader, train=True)
        val_loss,   val_acc   = run_epoch(val_loader,   train=False)
        scheduler.step(val_loss)
        elapsed = time.time() - t0
    
        history.append((train_loss, train_acc, val_loss, val_acc))
        print(f"  Epoch {epoch:>2}/{MAX_EPOCHS} | "
              f"train loss {train_loss:.4f} acc {train_acc*100:.2f}% | "
              f"val loss {val_loss:.4f} acc {val_acc*100:.2f}% | "
              f"{elapsed:.1f}s")
    
        # Early stopping
        if val_loss < best_val_loss - 1e-4:
            best_val_loss    = val_loss
            epochs_no_improv = 0
            torch.save(model.state_dict(), "best_fakenews_model.pt")
        else:
            epochs_no_improv += 1
            if epochs_no_improv >= PATIENCE:
                print(f"  Early stopping triggered after {epoch} epochs "
                      f"(no improvement for {PATIENCE} epochs).")
                break
    
    # Reload best checkpoint
    model.load_state_dict(torch.load("best_fakenews_model.pt"))
    print(f"\nBest validation loss: {best_val_loss:.4f}")
    
    
    # ── Evaluation ────────────────────────────────────────────────────────────────
    
    def evaluate(loader, split_name="Test"):
        model.eval()
        all_preds, all_probs, all_labels = [], [], []
    
        with torch.no_grad():
            for X_batch, y_batch in loader:
                X_batch = X_batch.to(DEVICE)
                probs   = model(X_batch).cpu().numpy()
                preds   = (probs >= 0.5).astype(int)
                all_probs.extend(probs)
                all_preds.extend(preds)
                all_labels.extend(y_batch.numpy().astype(int))
    
        all_labels = np.array(all_labels)
        all_preds  = np.array(all_preds)
        all_probs  = np.array(all_probs)
    
        auc = roc_auc_score(all_labels, all_probs)
        print(f"\n{'═'*60}")
        print(f"  {split_name} Evaluation")
        print(f"{'═'*60}")
        print(classification_report(all_labels, all_preds,
                                     target_names=["fake", "real"], digits=4))
        print(f"  ROC-AUC: {auc:.4f}")
        print(f"\n  Confusion Matrix (rows=actual, cols=predicted):")
        cm = confusion_matrix(all_labels, all_preds)
        print(f"              fake    real")
        print(f"  actual fake {cm[0,0]:>7,} {cm[0,1]:>7,}")
        print(f"  actual real {cm[1,0]:>7,} {cm[1,1]:>7,}")
        return all_labels, all_preds, all_probs

val_labels,  val_preds,  val_probs  = evaluate(val_loader,  "Validation")
test_labels, test_preds, test_probs = evaluate(test_loader, "Test")

Using device: cuda
Building TF-IDF features...
  TF-IDF done in 342.1s
  Feature matrix: 796,000 train | 99,500 val | 99,500 test | 30,000 features
  Label balance — train: 21.9% real | val: 22.2% real

Model architecture:
FakeNewsNet(
  (net): Sequential(
    (0): Linear(in_features=30000, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=1, bias=True)
    (9): Sigmoid()
  )
)
Trainable parameters: 15,427,585

Training...


C:\Users\Kian\miniconda3\envs\pytorch_env\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


NameError: name 'train_loader' is not defined

In [9]:
# ── 1. Imports & seeds ────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from scipy.sparse import issparse
import random
import time

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── 2. Hyperparameters ────────────────────────────────────────────────────────
VOCAB_SIZE   = 30_000
HIDDEN_1     = 512
HIDDEN_2     = 128
DROPOUT_1    = 0.4
DROPOUT_2    = 0.3
BATCH_SIZE   = 512
LR           = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS   = 20
PATIENCE     = 3
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# ── 3. Label binarisation ─────────────────────────────────────────────────────
REAL_LABELS = {"reliable", "mostly true"}

def binarize_labels(series):
    return (series.str.strip().str.lower().isin(REAL_LABELS)).astype(int)

# ── 4. TF-IDF features ────────────────────────────────────────────────────────
def tokens_to_string(token_list):
    if isinstance(token_list, list):
        return " ".join(token_list)
    return ""

print("Building TF-IDF features...")
t0 = time.time()

train_texts = train_df["tokens_stemmed"].apply(tokens_to_string)
val_texts   = val_df["tokens_stemmed"].apply(tokens_to_string)
test_texts  = test_df["tokens_stemmed"].apply(tokens_to_string)

tfidf = TfidfVectorizer(
    max_features = VOCAB_SIZE,
    sublinear_tf = True,
    min_df       = 10,
    ngram_range  = (1, 1),
)

X_train = tfidf.fit_transform(train_texts)
X_val   = tfidf.transform(val_texts)
X_test  = tfidf.transform(test_texts)

y_train = binarize_labels(train_df["type"])
y_val   = binarize_labels(val_df["type"])
y_test  = binarize_labels(test_df["type"])

print(f"  TF-IDF done in {time.time()-t0:.1f}s")
print(f"  Feature matrix: {X_train.shape[0]:,} train | "
      f"{X_val.shape[0]:,} val | {X_test.shape[0]:,} test | "
      f"{X_train.shape[1]:,} features")
print(f"  Label balance — train: {y_train.mean()*100:.1f}% real | "
      f"val: {y_val.mean()*100:.1f}% real")

# ── 5. Sparse Dataset & Loaders ───────────────────────────────────────────────
class SparseDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx].toarray().squeeze(0), dtype=torch.float32)
        return x, self.y[idx]

print("\nCreating datasets...")
train_ds = SparseDataset(X_train, y_train)
val_ds   = SparseDataset(X_val,   y_val)
test_ds  = SparseDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── 6. Model ──────────────────────────────────────────────────────────────────
class FakeNewsNet(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, drop1, drop2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(drop1),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(drop2),

            nn.Linear(hidden2, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model = FakeNewsNet(
    input_dim = VOCAB_SIZE,
    hidden1   = HIDDEN_1,
    hidden2   = HIDDEN_2,
    drop1     = DROPOUT_1,
    drop2     = DROPOUT_2,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel — trainable parameters: {total_params:,}")

# ── 7. Training ───────────────────────────────────────────────────────────────
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y_batch)
            correct    += ((preds >= 0.5).float() == y_batch).sum().item()
            total      += len(y_batch)
    return total_loss / total, correct / total

print("\nTraining...")
best_val_loss    = float("inf")
epochs_no_improv = 0

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    print(f"  Epoch {epoch:>2}/{MAX_EPOCHS} | "
          f"train loss {train_loss:.4f} acc {train_acc*100:.2f}% | "
          f"val loss {val_loss:.4f} acc {val_acc*100:.2f}% | "
          f"{elapsed:.1f}s")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss    = val_loss
        epochs_no_improv = 0
        torch.save(model.state_dict(), "best_fakenews_model.pt")
    else:
        epochs_no_improv += 1
        if epochs_no_improv >= PATIENCE:
            print(f"  Early stopping after {epoch} epochs.")
            break

model.load_state_dict(torch.load("best_fakenews_model.pt"))
print(f"\nBest validation loss: {best_val_loss:.4f}")

# ── 8. Evaluation ─────────────────────────────────────────────────────────────
def evaluate(loader, split_name="Test"):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            probs   = model(X_batch).cpu().numpy()
            preds   = (probs >= 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy().astype(int))

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    auc = roc_auc_score(all_labels, all_probs)
    print(f"\n{'═'*60}")
    print(f"  {split_name} Evaluation")
    print(f"{'═'*60}")
    print(classification_report(all_labels, all_preds,
                                 target_names=["fake", "real"], digits=4))
    print(f"  ROC-AUC: {auc:.4f}")
    print(f"\n  Confusion Matrix (rows=actual, cols=predicted):")
    cm = confusion_matrix(all_labels, all_preds)
    print(f"              fake    real")
    print(f"  actual fake {cm[0,0]:>7,} {cm[0,1]:>7,}")
    print(f"  actual real {cm[1,0]:>7,} {cm[1,1]:>7,}")

evaluate(val_loader,  "Validation")
evaluate(test_loader, "Test")

Using device: cuda
Building TF-IDF features...
  TF-IDF done in 361.5s
  Feature matrix: 796,000 train | 99,500 val | 99,500 test | 30,000 features
  Label balance — train: 21.9% real | val: 22.2% real

Creating datasets...

Model — trainable parameters: 15,427,585

Training...
  Epoch  1/20 | train loss 0.1132 acc 95.85% | val loss 0.0854 acc 96.83% | 660.2s
  Epoch  2/20 | train loss 0.0645 acc 97.63% | val loss 0.0805 acc 97.07% | 586.2s
  Epoch  3/20 | train loss 0.0434 acc 98.39% | val loss 0.0832 acc 97.14% | 557.2s
  Epoch  4/20 | train loss 0.0324 acc 98.79% | val loss 0.0872 acc 97.16% | 592.3s
  Epoch  5/20 | train loss 0.0275 acc 99.00% | val loss 0.0859 acc 97.23% | 181.3s
  Early stopping after 5 epochs.

Best validation loss: 0.0805

════════════════════════════════════════════════════════════
  Validation Evaluation
════════════════════════════════════════════════════════════
              precision    recall  f1-score   support

        fake     0.9781    0.9844    0.98

In [10]:
import pickle

# Save the PyTorch model weights
torch.save(model.state_dict(), "fakenews_model_final.pt")

# Save the TF-IDF vectorizer (needed to transform new text at inference time)
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Saved fakenews_model_final.pt and tfidf_vectorizer.pkl")

Saved fakenews_model_final.pt and tfidf_vectorizer.pkl


In [11]:
LIAR_DIR = r"C:\Users\Kian\Documents\GitHub\GDSProject\liar_dataset"

LIAR_COLS = [
    "id", "label", "statement", "subject", "speaker",
    "job", "state", "party", "barely_true_c", "false_c",
    "half_true_c", "mostly_true_c", "pants_fire_c", "context"
]

def load_liar(split):
    path = f"{LIAR_DIR}\\{split}.tsv"
    df = pd.read_csv(path, sep="\t", header=None, names=LIAR_COLS)
    return df

train_df = load_liar("train")
val_df   = load_liar("valid")   # LIAR uses "valid" not "val"
test_df  = load_liar("test")

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Label distribution:\n{train_df['label'].value_counts()}")

Train: 10,240 | Val: 1,284 | Test: 1,267
Label distribution:
label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64


In [14]:
VOCAB_SIZE   = 3_000
HIDDEN_1     = 64
HIDDEN_2     = 32
DROPOUT_1    = 0.5
DROPOUT_2    = 0.4
BATCH_SIZE   = 32
LR           = 1e-3
MAX_EPOCHS   = 50    # more epochs since each one is fast
PATIENCE     = 5

print(f"Train label balance: {y_train.mean()*100:.1f}% real")
print(f"Val label balance:   {y_val.mean()*100:.1f}% real")
print(f"Test label balance:  {y_test.mean()*100:.1f}% real")

Train label balance: 21.9% real
Val label balance:   22.2% real
Test label balance:  22.0% real
